In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import numpy as np
import pandas as pd

# Load the saved transformation matrix T
T = np.load('/content/drive/MyDrive/Ahsan/11. Cancer Data on 30th April/Reverse Calculation Related doc/transformation_matrix_T_High_class.npy')  # Update with the correct path
num_genes = T.shape[1]  # Number of genes in the original dataset

# Load original means and standard deviations
original_means = np.load('/content/drive/MyDrive/Ahsan/11. Cancer Data on 30th April/Reverse Calculation Related doc/original_means_High_class.npy')  # Update with the correct path
original_stds = np.load('/content/drive/MyDrive/Ahsan/11. Cancer Data on 30th April/Reverse Calculation Related doc/original_stds_High_class.npy')  # Update with the correct path

# Load synthetic genomap data
synthetic_genomap_path = r'/content/drive/MyDrive/Ahsan/11. Cancer Data on 30th April/synthetic_High_2490.npy' # Update with the correct path
synthetic_genomaps = np.load(synthetic_genomap_path)  # Shape: (num_samples, rowNum, colNum, 1)

# Define min and max values for reverse normalization
_MIN = -0.73866487 # Replace with the actual min value
_MAX = 37.5899   # Replace with the actual max value

# Reverse normalization for synthetic genomaps, for tanh purpose data = -1 + 2*(data - _MIN)/(_MAX - _MIN) was used
synthetic_genomaps = ((synthetic_genomaps + 1) * (_MAX - _MIN) / 2) + _MIN

# Inspect dimensions
print(f"Synthetic genomaps shape after reverse normalization: {synthetic_genomaps.shape}")

# Function to reverse the genomap to recover the original data
def reverse_genomap(genomaps, T, num_genes):
    num_samples, rowNum, colNum, _ = genomaps.shape
    totalGridPoint = rowNum * colNum

    # Scale T to match grid points
    projMat = T * totalGridPoint
    projMat_inv = np.linalg.pinv(projMat)  # Pseudoinverse of T

    # Flatten genomaps and project back to gene expression
    projM = np.zeros((num_samples, num_genes), dtype=np.float32)
    for i in range(num_samples):
        ex = genomaps[i, :, :, 0].flatten(order='F')  # Flatten in column-major order
        projM[i, :] = ex[:num_genes]

    # Reverse the projection to obtain the original data
    data = np.matmul(projM, projMat_inv)
    return data

# Reverse the synthetic genomaps
reversed_genomap = reverse_genomap(synthetic_genomaps, T, num_genes)

# Apply inverse normalization to the recovered data
recovered_data = (reversed_genomap * original_stds) + original_means

# Ensure float32 dtype for the recovered data
recovered_data = recovered_data.astype(np.float32)

# Convert the recovered gene expression data into a pandas DataFrame
recovered_data_df = pd.DataFrame(recovered_data)

# Save the recovered gene expression values to a CSV file
recovered_file_path = r'/content/drive/MyDrive/Ahsan/11. Cancer Data on 30th April/recovered_gene_expression_High_class_2490.csv'  # Update the path
recovered_data_df.to_csv(recovered_file_path, index=False, header=False, float_format='%.6f')

# Reshape recovered data to (1415, 40, 40, 1) for saving as .npy
recovered_npy = recovered_data.reshape(1415, 40, 40, 1)

# Save the reshaped recovered data as .npy
recovered_npy_path = r'/content/drive/MyDrive/Ahsan/11. Cancer Data on 30th April/recovered_gene_expression_High_class_2490.npy'  # Update the path
np.save(recovered_npy_path, recovered_npy)

print(f"Recovered gene expression data saved to: {recovered_file_path}")
print(f"Recovered gene expression data saved to: {recovered_npy_path} (NPY) with shape {recovered_npy.shape}")

Synthetic genomaps shape after reverse normalization: (1415, 40, 40, 1)
Recovered gene expression data saved to: /content/drive/MyDrive/Ahsan/11. Cancer Data on 30th April/recovered_gene_expression_High_class_2490.csv
Recovered gene expression data saved to: /content/drive/MyDrive/Ahsan/11. Cancer Data on 30th April/recovered_gene_expression_High_class_2490.npy (NPY) with shape (1415, 40, 40, 1)
